# Style2Fit — Step 4: Demo
Full pipeline demo with Gradio public URL.

**Files already in runtime:** `llm_adapter.zip` and `sdxl_lora.zip`

**Runtime:** A100 GPU.

In [ ]:
!pip install transformers peft bitsandbytes diffusers accelerate gradio -q

In [ ]:
import os
from google.colab import userdata
# Add HF_TOKEN via Colab Secrets (lock icon in left sidebar)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# Unzip adapter weights already in runtime
!unzip -q llm_adapter.zip
!unzip -q sdxl_lora.zip

# Handle double-nesting: if unzip created llm_adapter/llm_adapter/, flatten it
import os, shutil

def flatten_if_nested(outer, inner_name):
    inner = os.path.join(outer, inner_name)
    if os.path.isdir(inner):
        print(f'Flattening {inner} → {outer}')
        for f in os.listdir(inner):
            shutil.move(os.path.join(inner, f), os.path.join(outer, f))
        os.rmdir(inner)

flatten_if_nested('llm_adapter', 'llm_adapter')
flatten_if_nested('sdxl_lora',   'sdxl_lora')

print('llm_adapter:', os.listdir('llm_adapter'))
print('sdxl_lora:',   os.listdir('sdxl_lora'))

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from diffusers import StableDiffusionXLPipeline

BASE_LLM = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
BASE_SDXL = 'stabilityai/stable-diffusion-xl-base-1.0'

SYSTEM_PROMPT = """You are Style2Fit, a personal stylist assistant.
When someone describes their situation in casual language, you generate a complete,
coherent outfit recommendation in structured format.

Always respond with exactly:
Top: ...
Bottom: ...
Shoes: ...
Outerwear: ...
Accessories: ...
Aesthetic: ...
Explanation: ..."""

In [ ]:
# Load fine-tuned LLM
print('Loading LLM...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained('llm_adapter')
base = AutoModelForCausalLM.from_pretrained(BASE_LLM, quantization_config=bnb,
                                             device_map='auto', token=os.environ['HF_TOKEN'])
llm = PeftModel.from_pretrained(base, 'llm_adapter')
llm.eval()
print('LLM ready.')

In [ ]:
# Load fine-tuned SDXL
print('Loading SDXL...')
sdxl = StableDiffusionXLPipeline.from_pretrained(
    BASE_SDXL, torch_dtype=torch.bfloat16
).to('cuda')
sdxl.unet = PeftModel.from_pretrained(sdxl.unet, 'sdxl_lora').to(torch.bfloat16)
sdxl.enable_attention_slicing()
print('SDXL ready.')

In [ ]:
def generate_outfit_plan(situation, aesthetic=None):
    user_content = situation
    if aesthetic:
        user_content += f'\naesthetic: {aesthetic}'
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]
    encoded = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    )
    if hasattr(encoded, 'input_ids'):
        input_ids = encoded.input_ids.to(llm.device)
    else:
        input_ids = encoded.to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            input_ids, max_new_tokens=300, temperature=0.7,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)


def plan_to_image_prompt(plan_text):
    def extract(field):
        m = re.search(rf'{field}:\s*(.+?)(?:\n|$)', plan_text, re.IGNORECASE)
        return m.group(1).strip() if m else ''
    top        = extract('Top')
    bottom     = extract('Bottom')
    shoes      = extract('Shoes')
    outer      = extract('Outerwear')
    accessories = extract('Accessories')
    aesthetic  = extract('Aesthetic')

    pieces = [top, bottom, shoes]
    if outer and outer.lower() not in ('none', 'none needed', 'n/a'):
        pieces.append(outer)
    outfit_desc = ', '.join(p for p in pieces if p)

    # Accessories added as a separate strong clause so SDXL doesn't ignore them
    acc_clause = ''
    if accessories and accessories.lower() not in ('none', 'n/a', ''):
        acc_clause = f'Wearing {accessories}. '

    prompt = (
        f'Full body fashion editorial photo of a person wearing {outfit_desc}. '
        f'{acc_clause}'
        f'{aesthetic} aesthetic. '
        'Full length shot, head to toe, shoes clearly visible, '
        'soft natural lighting, clean white background, '
        'professional fashion photography, sharp focus, high resolution.'
    )
    negative_prompt = (
        'cropped, close up, portrait, headshot, cut off feet, cut off shoes, '
        'bad anatomy, deformed, extra limbs, blurry, low quality, '
        'cartoon, illustration, painting, drawing, unrealistic, nudity'
    )
    return prompt, negative_prompt


def run_pipeline(situation, aesthetic=None):
    plan = generate_outfit_plan(situation, aesthetic)
    prompt, negative_prompt = plan_to_image_prompt(plan)
    image = sdxl(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30, guidance_scale=7.5,
        height=1024, width=768,
    ).images[0]
    return plan, image


print('Pipeline functions ready.')

In [ ]:
# ════════════════════════════════════════════════════════
# EVALUATION — run this single cell for all metrics
# ════════════════════════════════════════════════════════
!pip install rouge-score -q

import json, random
import numpy as np
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer
from collections import Counter
from tqdm.notebook import tqdm

def _encode(msgs, model):
    encoded = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt')
    if hasattr(encoded, 'input_ids'):
        return encoded.input_ids.to(model.device)
    return encoded.to(model.device)

# ── 1. Load 30 held-out eval pairs ───────────────────────
with open('train.jsonl') as f:
    all_pairs = [json.loads(l) for l in f]
random.seed(42)
eval_pairs = random.sample(all_pairs, min(30, len(all_pairs)))

REQUIRED_FIELDS = ['top', 'bottom', 'shoes', 'outerwear', 'accessories', 'aesthetic', 'explanation']
NONE_VALUES = {'none', 'none needed', 'n/a', 'na', ''}

def extract_field(text, field):
    m = re.search(rf'{field}:\s*(.+?)(?:\n|$)', text, re.IGNORECASE)
    return m.group(1).strip() if m else ''

def check_format(text):
    return all(f in text.lower() for f in REQUIRED_FIELDS)

print(f'Eval set: {len(eval_pairs)} pairs')

# ── 2. Run fine-tuned model on all 30 prompts ────────────
def _generate_tuned(situation):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': situation}]
    input_ids = _encode(msgs, llm)
    with torch.no_grad():
        out = llm.generate(input_ids, max_new_tokens=300, temperature=0.7,
                           top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

predictions = []
for pair in tqdm(eval_pairs, desc='Fine-tuned model inference'):
    predictions.append(_generate_tuned(pair['instruction']))
print(f'{len(predictions)} predictions done.')

# ── 3. Format compliance + ROUGE-L ───────────────────────
scorer_r = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
format_scores, rouge_scores = [], []
for pred, pair in zip(predictions, eval_pairs):
    format_scores.append(1 if check_format(pred) else 0)
    rouge_scores.append(scorer_r.score(pair['output'], pred)['rougeL'].fmeasure)

format_compliance = np.mean(format_scores) * 100
mean_rouge = np.mean(rouge_scores)
print(f'Format compliance : {format_compliance:.1f}%')
print(f'ROUGE-L mean      : {mean_rouge:.3f}')

# ── 4. Field completeness + Aesthetic diversity ───────────
FIELDS = ['Top', 'Bottom', 'Shoes', 'Outerwear', 'Accessories', 'Aesthetic', 'Explanation']
field_fill = {f: [] for f in FIELDS}
aesthetics_seen = []
for pred in predictions:
    for f in FIELDS:
        val = extract_field(pred, f + ':').lower().strip()
        field_fill[f].append(0 if val in NONE_VALUES else 1)
    aes = extract_field(pred, 'Aesthetic:').lower().strip()
    if aes:
        aesthetics_seen.append(aes)
field_pct = {f: np.mean(v) * 100 for f, v in field_fill.items()}
aes_counter = Counter(aesthetics_seen)
print(f'Aesthetic diversity: {len(aes_counter)} distinct across 30 prompts')

# ── 5. Before vs After (base model, no adapter) ───────────
COMPARE_PROMPTS = [
    'i have a coffee date tmrw what do i wear',
    'first day at my internship, business casual',
    'going to a rooftop bar friday night',
    'omg i have a presentation today help',
    'beach vacation next week, need full looks',
]
print('\nLoading BASE model for before/after comparison...')
bnb2 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.bfloat16)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_LLM, quantization_config=bnb2, device_map='auto', token=os.environ['HF_TOKEN'])
base_model.eval()

def _generate_base(prompt):
    msgs = [{'role': 'user', 'content': f'What should I wear? {prompt}'}]
    input_ids = _encode(msgs, base_model)
    with torch.no_grad():
        out = base_model.generate(input_ids, max_new_tokens=200, temperature=0.7,
                                  top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

before_after = []
print('\n' + '='*70)
for prompt in COMPARE_PROMPTS:
    base_out  = _generate_base(prompt)
    tuned_out = _generate_tuned(prompt)
    before_after.append({'prompt': prompt, 'base': base_out, 'tuned': tuned_out})
    print(f'PROMPT : "{prompt}"')
    print(f'BASE   [structured={check_format(base_out)}]  {base_out[:180].strip()}...')
    print(f'TUNED  [structured={check_format(tuned_out)}]  {tuned_out[:180].strip()}...')
    print('─'*70)

base_format = sum(check_format(r['base']) for r in before_after) / len(before_after) * 100

# ── 6. Summary dashboard ─────────────────────────────────
ROSE, BLUSH, TAUPE, BROWN = '#C08080', '#E8D5CB', '#9E8E82', '#6B5B52'
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#FAF8F5')

ax = axes[0]
ax.set_facecolor('#FAF8F5')
bars = ax.bar(['Base\nLLaMA 3.1', 'Style2Fit\n(fine-tuned)'],
              [base_format, format_compliance], color=[BLUSH, ROSE], width=0.45, zorder=3)
ax.set_ylim(0, 115)
ax.set_title('Structured Output Rate', color=BROWN, fontsize=12, pad=12)
for bar, val in zip(bars, [base_format, format_compliance]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}%', ha='center', va='bottom', color=BROWN, fontsize=13, fontweight='bold')
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(colors=TAUPE)
ax.grid(axis='y', color=BLUSH, linestyle='--', linewidth=0.7, zorder=0)

ax = axes[1]
ax.set_facecolor('#FAF8F5')
ax.hist(rouge_scores, bins=12, color=ROSE, alpha=0.85, edgecolor='white', zorder=3)
ax.axvline(mean_rouge, color=BROWN, linestyle='--', linewidth=1.5, label=f'Mean = {mean_rouge:.3f}')
ax.set_xlabel('ROUGE-L Score', color=TAUPE, fontsize=10)
ax.set_ylabel('Count', color=TAUPE, fontsize=10)
ax.set_title('ROUGE-L vs Ground Truth\n(30 held-out prompts)', color=BROWN, fontsize=12, pad=12)
ax.legend(frameon=False, labelcolor=BROWN, fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.tick_params(colors=TAUPE)
ax.grid(axis='y', color=BLUSH, linestyle='--', linewidth=0.7, zorder=0)

ax = axes[2]
ax.set_facecolor('#FAF8F5')
fields = list(field_pct.keys())
values = list(field_pct.values())
bars2 = ax.barh(fields, values, color=[ROSE if v >= 90 else BLUSH for v in values], zorder=3)
ax.set_xlim(0, 115)
ax.set_xlabel('Fields Populated (%)', color=TAUPE, fontsize=10)
ax.set_title('Field Completeness', color=BROWN, fontsize=12, pad=12)
for bar, val in zip(bars2, values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', color=BROWN, fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.tick_params(colors=TAUPE)
ax.grid(axis='x', color=BLUSH, linestyle='--', linewidth=0.7, zorder=0)

plt.suptitle('Style2Fit — Evaluation Summary', fontsize=14, color=BROWN, y=1.02)
plt.tight_layout()
plt.savefig('eval_summary.png', dpi=150, bbox_inches='tight', facecolor='#FAF8F5')
plt.show()

# ── 7. Save all results to disk + download ───────────────
eval_results = {
    'summary': {
        'format_compliance_finetuned': round(format_compliance, 2),
        'format_compliance_base':      round(base_format, 2),
        'rouge_l_mean':                round(float(mean_rouge), 4),
        'rouge_l_median':              round(float(np.median(rouge_scores)), 4),
        'rouge_l_std':                 round(float(np.std(rouge_scores)), 4),
        'aesthetic_diversity':         len(aes_counter),
        'eval_set_size':               len(eval_pairs),
        'training_pairs':              540,
        'base_model':                  'LLaMA 3.1 8B Instruct',
        'finetuning':                  'QLoRA r=16 alpha=32 3 epochs',
    },
    'field_completeness': {f: round(v, 2) for f, v in field_pct.items()},
    'top_aesthetics':     dict(aes_counter.most_common(10)),
    'rouge_scores':       [round(s, 4) for s in rouge_scores],
    'before_after':       before_after,
    'predictions': [
        {'prompt': p['instruction'], 'ground_truth': p['output'], 'prediction': pred}
        for p, pred in zip(eval_pairs, predictions)
    ],
}

with open('eval_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)
print('Saved eval_results.json')

from google.colab import files
files.download('eval_results.json')
files.download('eval_summary.png')
print('Downloaded eval_results.json + eval_summary.png')

print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  STYLE2FIT — EVALUATION SUMMARY')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Format compliance — fine-tuned : {format_compliance:.1f}%')
print(f'  Format compliance — base model : {base_format:.1f}%')
print(f'  ROUGE-L (mean)                 : {mean_rouge:.3f}')
print(f'  Aesthetic diversity            : {len(aes_counter)} distinct / 30 prompts')
print(f'  Training set                   : 540 pairs (500 fashion200k + 40 synthetic)')
print(f'  Fine-tuning                    : QLoRA  r=16  α=32  3 epochs')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

In [ ]:
# Launch Gradio demo with public URL (for pitch day)
import gradio as gr

AESTHETICS = [
    'infer from situation', 'clean girl', 'dark academia', 'old money',
    'streetwear', 'soft girl', 'indie', 'minimalist', 'boho', 'preppy', 'y2k',
]

BASE_SYSTEM_SIMPLE = 'You are a helpful fashion assistant.'

def _encode_gradio(msgs, model):
    encoded = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt')
    if hasattr(encoded, 'input_ids'):
        return encoded.input_ids.to(model.device)
    return encoded.to(model.device)

def generate_base_gradio(situation):
    """Base LLaMA with no system prompt and no fine-tuning."""
    msgs = [
        {'role': 'system', 'content': BASE_SYSTEM_SIMPLE},
        {'role': 'user', 'content': f'What should I wear? {situation}'},
    ]
    input_ids = _encode_gradio(msgs, llm.base_model)
    with torch.no_grad():
        out = llm.base_model.generate(
            input_ids, max_new_tokens=250, temperature=0.7,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

def gradio_fn(situation, aesthetic):
    if not situation.strip():
        return 'Please enter a situation.', None
    aes = None if aesthetic == 'infer from situation' else aesthetic
    plan, image = run_pipeline(situation, aes)
    return plan.replace('\n', '\n\n'), image

def gradio_compare_fn(situation):
    if not situation.strip():
        return 'Please enter a situation.', ''
    base_out   = generate_base_gradio(situation)
    tuned_out  = generate_outfit_plan(situation)
    base_fmt   = '✅ structured' if all(f in base_out.lower() for f in ['top:', 'bottom:', 'shoes:']) else '❌ unstructured'
    tuned_fmt  = '✅ structured' if all(f in tuned_out.lower() for f in ['top:', 'bottom:', 'shoes:']) else '❌ unstructured'
    base_label  = f'**Base LLaMA 3.1 8B** ({base_fmt})\n\n{base_out}'
    tuned_label = f'**Style2Fit (fine-tuned)** ({tuned_fmt})\n\n{tuned_out.replace(chr(10), chr(10)+chr(10))}'
    return base_label, tuned_label

with gr.Blocks(title='Style2Fit', theme=gr.themes.Soft(primary_hue='rose')) as demo:
    gr.Markdown('# Style2Fit ✨\n### Describe your situation. Get a real outfit. See it on a person.')

    with gr.Tabs():
        # ── Tab 1: Full pipeline ──────────────────────────────
        with gr.Tab('Generate Outfit'):
            with gr.Row():
                with gr.Column():
                    situation_box = gr.Textbox(
                        label="What's the situation?",
                        placeholder='i have a coffee date tmrw what do i wear',
                        lines=2,
                    )
                    aesthetic_box = gr.Dropdown(
                        choices=AESTHETICS, value='infer from situation',
                        label='Aesthetic (optional)',
                    )
                    btn = gr.Button('Generate Outfit', variant='primary')
                    gr.Examples(
                        examples=[
                            ['i have a coffee date tmrw what do i wear', 'infer from situation'],
                            ['first day at my internship, business casual', 'infer from situation'],
                            ['concert this weekend, indie/alt vibe', 'indie'],
                            ['birthday dinner at a nice restaurant', 'old money'],
                            ['omg i have a presentation today help', 'infer from situation'],
                            ['packing for lisbon for a week in april', 'infer from situation'],
                        ],
                        inputs=[situation_box, aesthetic_box],
                    )
                with gr.Column():
                    plan_out  = gr.Textbox(label='Outfit Plan', lines=10)
                    image_out = gr.Image(label='Outfit Visual', type='pil')
            btn.click(gradio_fn, inputs=[situation_box, aesthetic_box], outputs=[plan_out, image_out])
            situation_box.submit(gradio_fn, inputs=[situation_box, aesthetic_box], outputs=[plan_out, image_out])

        # ── Tab 2: Before vs After ────────────────────────────
        with gr.Tab('Before vs After'):
            gr.Markdown('### Base LLaMA 3.1 8B vs Style2Fit fine-tuned\nSame prompt, same model — only the fine-tuning adapter changes.')
            compare_input = gr.Textbox(
                label="What's the situation?",
                placeholder='i have a coffee date tmrw what do i wear',
                lines=2,
            )
            compare_btn = gr.Button('Compare', variant='primary')
            gr.Examples(
                examples=[
                    ['i have a coffee date tmrw what do i wear'],
                    ['first day at my internship, business casual'],
                    ['going to a rooftop bar friday night'],
                    ['omg i have a presentation today help'],
                    ['beach vacation next week, need full looks'],
                ],
                inputs=[compare_input],
            )
            with gr.Row():
                base_out_box  = gr.Markdown(label='Base model')
                tuned_out_box = gr.Markdown(label='Fine-tuned')
            compare_btn.click(gradio_compare_fn, inputs=[compare_input],
                              outputs=[base_out_box, tuned_out_box])
            compare_input.submit(gradio_compare_fn, inputs=[compare_input],
                                 outputs=[base_out_box, tuned_out_box])

demo.launch(share=True)